In [1]:
import geolipi.symbolic as gls
import sysl.symbolic as csls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader.shader_module import SMMap
from IPython.display import display, HTML
from sysl.shader_vis.generate_shader_html import create_shader_html, make_jupyter_compatible_html

settings = {
    "render_mode": "v1",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 1,
        "_RAYCAST_MAX_STEPS": 200,
    },
    "set_to_ubo": False,
    "export_params": False,

}


In [2]:
import numpy as np

def generate_sdf_volume(resolution=32, radius=0.5, filename='sdf_volume.npy'):
    # Define grid
    lin = np.linspace(-1, 1, resolution)
    x, y, z = np.meshgrid(lin, lin, lin, indexing='ij')  # shape: (res, res, res)
    pos = np.stack([x, y, z], axis=-1)  # shape: (res, res, res, 3)

    # Compute signed distance to sphere
    sdf = np.linalg.norm(pos, axis=-1) - radius  # shape: (res, res, res)

    # Save
    np.save(filename, sdf.astype(np.float32))
    print(f"SDF saved to {filename} with shape {sdf.shape}")

generate_sdf_volume()

SDF saved to sdf_volume.npy with shape (32, 32, 32)


In [3]:
import geolipi.symbolic as gls
import sysl.symbolic as sls
from sysl.shader.evaluate import evaluate_to_shader
from sysl.shader_vis.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from IPython.display import display, HTML

# Create basic shapes
sphere = gls.Sphere3D((1.0,))
box = gls.Cuboid3D((2, 0.01, 2,))

# Combine with operations
# scene = gls.Translate3D(
#     gls.Union(gls.Scale3D(sphere, (0.5, 0.5, 0.5)), box),
#     (0.5, 0.5, 0)
# )

# Assign materials
# material = cls.NonEmissiveMaterialV3(
#     (1.0, 0.0, 0.0), 
#     (1.0,), (1.0,), (0.3,)
# )
material = sls.SMPLMaterial((2.0,))
scene_with_material = sls.MatSolidV1(sphere, material)

# Render
shader_code, uniforms, textures = evaluate_to_shader(scene_with_material, settings=settings)

with open("test.glsl", "w") as f:
    f.write(shader_code)

# TO visualize in a browser:
html_code = create_shader_html(shader_code, uniforms, textures, show_controls=True)
with open("test.html", "w") as f:
    f.write(html_code)

# To visualize inline in jupyter notebook:
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:
# Create basic shapes
floor = cls.MatSolidV3(
    gls.Cuboid3D((4, 0.05, 4,)),
    cls.MatReference("MatWood")
)

obj_1 = cls.MatSolidV3(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.0, 0.5, 0.0)),
    cls.NonEmissiveMaterialV3(
        (1.0, 0.0, 0.0), 
        (1.0,), (.0,), (0.,)
    )
)

obj_2 = cls.MatSolidV3(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.5, 0.5, 0.5)),
    cls.NonEmissiveMaterialV3(
        (0.0, 0.0, 1.0), 
        (0.0,), (1.0,), (0.0,)
    )
)

obj_3 = cls.MatSolidV3(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.0, 0.5, 0.5)),
    cls.NonEmissiveMaterialV3(
        (0.0, 1.0, 0.0), 
        (0.0,), (1.0,), (0.9,)
    )
)
obj_4 = cls.MatSolidV3(
    gls.Translate3D(gls.Sphere3D((0.2,)), (-0.5, 0.5, 0.5)),
    cls.NonEmissiveMaterialV3(
        (0.7, 1.0, 0.0), 
        (1.0,), (0.0,), (0.3,)
    )
)
settings = {
    "render_mode": "v3",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 4,
        "_RAYCAST_MAX_STEPS": 400,
    }
}


scene = gls.Union(floor, 
gls.Translate3D(gls.Union(obj_1, obj_2, obj_3, obj_4), 
# (0.0, 0.0, 0.0)
gls.UniformVec3((-0.5, -0.5, -0.5), (0.0, 0.0, 0.0), (0.5, 0.5, 0.5), "translate")
))
# Render
shader_code, uniforms = evaluate_to_shader(scene, settings)
html_code = create_shader_html(shader_code, uniforms, show_controls=True)

# To visualize inline in jupyter notebook:
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:
# Create basic shapes
floor = cls.MatSolidV2(
    gls.Cuboid3D((2, 0.05, 2,)),
    cls.RGBMaterial((0.3, 0.3, 1.0))
)

obj_1 = cls.MatSolidV2(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.0, 0.5, 0.0)),
    cls.RGBMaterial(
        (1.0, 0.0, 0.0), 
        (1.0,), (.0,), (0.,)
    )
)

obj_2 = cls.MatSolidV2(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.5, 0.5, 0.5)),
    cls.RGBMaterial(
        (0.0, 0.0, 1.0), 
    )
)

obj_3 = cls.MatSolidV2(
    gls.Translate3D(gls.Sphere3D((0.2,)), (0.0, 0.5, 0.5)),
    cls.RGBMaterial(
        (0.0, 1.0, 0.0), 
    )
)
obj_4 = cls.MatSolidV2(
    gls.Translate3D(gls.Sphere3D((0.2,)), (-0.5, 0.5, 0.5)),
    cls.RGBMaterial(
        (0.7, 1.0, 0.0), 
    )
)
settings = {
    "render_mode": "v2",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": False,
        "_AA": 4,
        "_RAYCAST_MAX_STEPS": 400,
    }
}


scene = gls.Union(floor, obj_1, obj_2, obj_3, obj_4)
# Render
shader_code, uniforms = evaluate_to_shader(scene, settings)
html_code = create_shader_html(shader_code, uniforms, show_controls=True)


# To visualize inline in jupyter notebook:
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:

import cisl.symbolic as csls
solid_expr = gls.Sphere3D((0.5,))
obj_1 = csls.MatSolidV3(solid_expr, csls.MatReference("MatMoss"))
obj_2 = csls.MatSolidV3(gls.Translate3D(solid_expr, (0.0, 0.5, 0.0)), csls.MatReference("MatGold"))
obj_3 = csls.MatSolidV3(gls.Translate3D(solid_expr, (0.0, 0.25, 0.5)), csls.MatReference("MatRustyPaint"))
expression = gls.SmoothUnion(obj_1, obj_2, (0.2,))
expression = csls.MatColorOnly(expression, obj_3)

# Render
shader_code, uniforms = evaluate_to_shader(expression)
html_code = create_shader_html(shader_code, uniforms, show_controls=True)

# To visualize inline in jupyter notebook:
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:
# Create basic shapes

solid_expr = gls.Sphere3D((0.5,))
obj_1 = csls.MatSolidV2(solid_expr, csls.RGBMaterial((0., 1.0, 0.0)))
obj_2 = csls.MatSolidV2(gls.Translate3D(solid_expr, (0.0, 0.5, 0.0)), csls.RGBMaterial((0., 0.0, 1.0)))
obj_3 = csls.MatSolidV2(gls.Translate3D(solid_expr, (0.0, 0.25, 0.5)), csls.RGBMaterial((1.0, 0.0, 0.0)))
expression = gls.Union(obj_1, obj_2)
expression = csls.MatSmoothColorOnly(expression, obj_3, 
    gls.UniformFloat((0.0,), (0.2,), (1.0,), "k")
    )

# Render
shader_code, uniforms = evaluate_to_shader(expression, settings)
html_code = create_shader_html(shader_code, uniforms, show_controls=True)

# To visualize inline in jupyter notebook:
jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:
# solid_expr = gls.Union(
# gls.Translate3D(gls.Cuboid3D((0.72, 0.4, 0.2)), (0.0, 0.5, 0.0)),
# gls.Translate3D(gls.Sphere3D((0.2,)), (0.25, 0.5, 0.25)), #(0.5,)
# )
# solid_expr = gls.Sphere3D((0.5,))
# solid_expr_2 = gls.Cuboid3D((0.72, 0.4, 0.2))
# # solid_expr = gls.EulerRotate3D(solid_expr, (0.0, 1.0, 0.5))
# mat_ctrl = gls.UniformFloat((0.0,), (2.0,), (10.0,), "mat")
# expression_1 = csls.MatSolidV1(solid_expr, 
#                 csls.SMPLMaterial(
#                     gls.BinaryOperator(
#                         mat_ctrl,
#                         gls.Float((1.4,)),
#                         "add"
#                     )
#                     # (3.5,)
#                     ))

# expression_2 = csls.MatSolidV1(gls.Translate3D(solid_expr_2, (0.4, 0.3, 0.0)), 
#             csls.SMPLMaterial(
#                     mat_ctrl
#                     # (4.5,)
#                 ))

# # expression = csls.MatSolid(gls.Difference(solid_expr, gls.Translate3D(solid_expr, (0.4, 0.0,0.0))), csls.SMPLMaterial((2.5,)))
# expression = gls.SmoothDifference(expression_1, expression_2, 
# # (0.1,)
# gls.UniformFloat((0.0,), (0.0,), (1.0,), "sm_rate")
# )
# # expression = gls.Difference(expression_1, expression_2,)
# expression = gls.Translate3D(
#     expression,
#     # csls.MatSolid(expression, csls.SMPLMaterial(
#     #     gls.UniformFloat((0.0,), (0.0,), (10.0,), "k")
#     # )),
#     # gls.UniformVec3((0.0, 0.0, 0.0), (0.2, 0.2, 0.2), (1.0, 1.0, 1.0), "t")
#     (0.0, 0.4, 0.0)
# )
solid_expr = gls.Sphere3D((0.5,))
obj_1 = csls.MatSolidV2(solid_expr, csls.RGBMaterial((0., 1.0, 0.0)))
obj_2 = csls.MatSolidV2(gls.Translate3D(solid_expr, (0.0, 0.5, 0.0)), csls.RGBMaterial((0., 0.0, 1.0)))
obj_3 = csls.MatSolidV2(gls.Translate3D(solid_expr, (0.0, 0.25, 0.5)), csls.RGBMaterial((1.0, 0.0, 0.0)))
expression = gls.SmoothUnion(obj_1, obj_2, (0.0,))
# expression = csls.MatColorOnly(expression, obj_3)
expression = csls.MatSmoothColorOnly(expression, obj_3, 
    # (0.1,)
    gls.UniformFloat((0.0,), (0.2,), (1.0,), "k")
    )
expression = csls.BoundedSolid(expression, gls.Cuboid3D((2.0, 2.0, 2.0)), )
# expression = gls.Union(obj_1, obj_2)
# expression = gls.SmoothUnion(expression, obj_3, (0.1,))

solid_expr = gls.Sphere3D((0.5,))
expression = csls.MatSolidV3(solid_expr, 
                csls.NonEmissiveMaterialV3(
                    (1.0, 0.0, 0.0), 
                    # gls.UniformVec3((0.0, 0.0, 0.0), (0.0, 0.0, 1.0), (1.0, 1.0, 1.0), "albedo"),
                    # (0.0, 0.0, 0.0), 
                    # gls.UniformVec3((0.0, 0.0, 0.0), (0.0, 0.0, 0.0), (0.0, 0.0, 0.0), "emissive"),
                    (1.0,), 
                    # gls.UniformFloat((0.0,), (0.7,), (1.0,), "roughness"),
                    (1.0,), 
                    # gls.UniformFloat((0.0,), (0.7,), (1.0,), "clearcoat"),
                    (0.3,)
                    # gls.UniformFloat((0.0,), (0.7,), (1.0,), "metallic"),
                    )
                    )


solid_expr_2 = gls.Translate3D(gls.Cuboid3D((10.72, 0.2, 10.7)), (0.0, -1.0, 0.0))
expression_2 = csls.MatSolidV3(solid_expr_2, 
                csls.MatReference("MatFloor")
                # csls.MaterialV3(
                #     (0.0, 0.0, 1.0), 
                #     # gls.UniformVec3((0.0, 0.0, 0.0), (0.0, 0.0, 1.0), (1.0, 1.0, 1.0), "albedo"),
                #     (0.0, 0.0, 0.0), 
                #     # gls.UniformVec3((0.0, 0.0, 0.0), (0.0, 0.0, 0.0), (0.0, 0.0, 0.0), "emissive"),
                #     (0.0,), 
                #     # gls.UniformFloat((0.0,), (0.7,), (1.0,), "roughness"),
                #     (0.0,), 
                #     # gls.UniformFloat((0.0,), (0.7,), (1.0,), "clearcoat"),
                #     (0.0,)
                #     # gls.UniformFloat((0.0,), (0.7,), (1.0,), "metallic"),
                #     )
                    )
expression = gls.Union(expression, expression_2)
# expression = csls.MatSolidV3(solid_expr, csls.NonEmissiveMaterialV3((1.0, 0.0, 0.0), (0.0,), (0.0,), (0.0,)))
# expression = csls.MatSolidV3(solid_expr, csls.MatReference("name"))


In [ ]:
shader_code, uniforms, shader_context = evaluate_to_shader(expression, settings, return_shader_context=True)
# Smpl front end coder. 
html_code = create_shader_html(shader_code, uniforms, show_controls=True)
with open("test.html", "w") as f:
    f.write(html_code)


In [ ]:
from cisl.shader.evaluate import evaluate_to_shader
from cisl.shader_vis.offline_render import render_cisl_shader_to_numpy

# output = render_cisl_shader_to_numpy(shader_code, uniforms)

In [ ]:

jupy_wrapper_html = make_jupyter_compatible_html(html_code)
display(HTML(jupy_wrapper_html))

In [ ]:
print(shader_context.shader_modules['SCENE_EXPRESSION'].code)

In [ ]:
# Sample a random program from CSG2D/CSG3D

# Get the shader code. 

# -> Implies Get Code -> Apply basic RGB Material. 

# First goal -> render arbitary 3D CSG expression from Geolipi using CISL. 
# -> requirements: a) SDF map. b) Uniform, variable, operators handing, c) basic material, d) basic scene handling, e) render config handing.
# Just render the SDF -> Apply strict materials. 

# then we go for smooth material operators with a material eval in the second thing. 

# basically register geometry. 

import geolipi.symbolic as gls
import cisl.symbolic as csls
from cisl.shader.evaluate import evaluate_to_shader
from cisl.shader.shader_module import SMMap
import json
settings = {
    "render_mode": "default",
}

def convert_to_minimal_html(shader_code, uniforms):
    # Convert the shader code into a minimal HTML file.
    # The HTML file should have a canvas element with the shader code and the uniforms.
    # The uniforms should be added as sliders.
    # The HTML file should be saved to the current directory.
    # Have a REGL based viewer. 
    # Based on the uniforms add sliders.
    # For camera Etc. already existing controls okay. 
    
    # Generate uniform sliders HTML
    uniform_controls = ""
    uniform_js_setup = ""
    
    for name, value in uniforms.items():
        if isinstance(value, (int, float)):
            # Create slider for numeric uniforms
            uniform_controls += f'''
    <div class="control">
        <label>{name}: <span id="{name}-value">{value}</span></label>
        <input type="range" id="{name}-slider" min="-2" max="2" step="0.01" value="{value}">
    </div>'''
            uniform_js_setup += f'''
    const {name}Slider = document.getElementById('{name}-slider');
    const {name}Value = document.getElementById('{name}-value');
    let {name} = {value};
    
    {name}Slider.addEventListener('input', (e) => {{
        {name} = parseFloat(e.target.value);
        {name}Value.textContent = {name}.toFixed(2);
    }});
    '''
    
    # Create the minimal HTML template
    html_template = f'''<!DOCTYPE html>
<html>
<head>
    <title>CISL Shader Preview</title>
    <style>
        body {{
            margin: 0;
            padding: 20px;
            font-family: Arial, sans-serif;
            background: #222;
            color: #fff;
        }}
        
        #container {{
            display: flex;
            gap: 20px;
        }}
        
        canvas {{
            border: 1px solid #555;
            background: #000;
        }}
        
        #controls {{
            width: 250px;
            padding: 20px;
            background: #333;
            border-radius: 5px;
        }}
        
        .control {{
            margin-bottom: 15px;
        }}
        
        label {{
            display: block;
            margin-bottom: 5px;
            font-size: 14px;
        }}
        
        input[type="range"] {{
            width: 100%;
        }}
        
        h3 {{
            margin-top: 0;
            color: #4CAF50;
        }}
    </style>
</head>
<body>
    <div id="container">
        <canvas id="canvas" width="800" height="600"></canvas>
        <div id="controls">
            <h3>Uniforms</h3>
            {uniform_controls}
        </div>
    </div>

    <script src="https://cdn.jsdelivr.net/npm/regl@2.1.0/dist/regl.min.js"></script>
    <script>
        const canvas = document.getElementById('canvas');
        const regl = require('regl')(canvas);
        
        // Camera state
        let camera = {{
            eye: [0, 0, 5],
            center: [0, 0, 0],
            up: [0, 1, 0]
        }};
        
        // Set up uniform controls
        {uniform_js_setup}
        
        // Vertex shader
        const vertexShader = `
        attribute vec2 position;
        void main() {{
            gl_Position = vec4(position, 0.0, 1.0);
        }}
        `;
        
        // Fragment shader (embedded from CISL)
        const fragmentShader = `
        precision mediump float;
        uniform vec2 u_resolution;
        uniform float u_time;
        {shader_code}
        
        void main() {{
            vec2 uv = gl_FragCoord.xy / u_resolution;
            uv = uv * 2.0 - 1.0;
            uv.x *= u_resolution.x / u_resolution.y;
            
            gl_FragColor = vec4(render(uv), 1.0);
        }}
        `;
        
        // Create REGL command
        const drawShader = regl({{
            frag: fragmentShader,
            vert: vertexShader,
            attributes: {{
                position: [[-1, -1], [1, -1], [-1, 1], [1, 1]]
            }},
            uniforms: {{
                u_resolution: [canvas.width, canvas.height],
                u_time: regl.context('time'),
                ...Object.fromEntries(Object.keys({json.dumps(uniforms)}).map(key => [key, () => eval(key)]))
            }},
            count: 4,
            primitive: 'triangle strip'
        }});
        
        // Render loop
        regl.frame(() => {{
            regl.clear({{
                color: [0, 0, 0, 1]
            }});
            
            drawShader();
        }});
        
        // Basic mouse controls for camera
        let mouseDown = false;
        let lastMouseX = 0;
        let lastMouseY = 0;
        
        canvas.addEventListener('mousedown', (e) => {{
            mouseDown = true;
            lastMouseX = e.clientX;
            lastMouseY = e.clientY;
        }});
        
        canvas.addEventListener('mouseup', () => {{
            mouseDown = false;
        }});
        
        canvas.addEventListener('mousemove', (e) => {{
            if (!mouseDown) return;
            
            const deltaX = e.clientX - lastMouseX;
            const deltaY = e.clientY - lastMouseY;
            
            // Simple rotation logic (can be improved)
            camera.eye[0] += deltaX * 0.01;
            camera.eye[1] += deltaY * 0.01;
            
            lastMouseX = e.clientX;
            lastMouseY = e.clientY;
        }});
        
        // Zoom with mouse wheel
        canvas.addEventListener('wheel', (e) => {{
            e.preventDefault();
            const zoom = e.deltaY * 0.001;
            camera.eye[2] += zoom;
            if (camera.eye[2] < 0.1) camera.eye[2] = 0.1;
            if (camera.eye[2] > 20) camera.eye[2] = 20;
        }});
    </script>
</body>
</html>'''
    
    return html_template

expression = csls.MatSolid(gls.Cuboid3D((0.2, 0.4, 0.2)), csls.SMPLMaterial((1.5)))

shader_code, uniforms = evaluate_to_shader(expression, settings)
# Convert it into simple front end. 
html_code = convert_to_minimal_html(shader_code, uniforms)


In [ ]:
file_name = "/home/colligo/projects/rejig/html_gen/generated_shader_generated.html"
html_code = open(file_name, 'r').read()

In [ ]:
from IPython.display import display, HTML
import tempfile

# 1. Save the full HTML code to a temp file
# html_code = f"<!DOCTYPE html> {html_code}"  # your full HTML string here

# 2. Optional: escape quotes if using srcdoc
import html
escaped_html = html.escape(html_code)

# 3. Build iframe for inline rendering
iframe_html = f"""
<iframe
    srcdoc="{escaped_html}"
    style="width: 100%; height: 800px; border: none;"
    sandbox="allow-scripts allow-same-origin"
></iframe>
"""

display(HTML(iframe_html))

In [ ]:
# 2. Optional: escape quotes if using srcdoc
import html
escaped_html = html.escape(html_code)

# 3. Build iframe for inline rendering
iframe_html = f"""
<iframe
    srcdoc="{escaped_html}"
    style="width: 100%; height: 800px; border: none;"
    sandbox="allow-scripts allow-same-origin"
></iframe>
"""

display(HTML(iframe_html))